<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/849_HITLv2_Audit_Feedback.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Audit & Feedback Module

## What This Module Does

This module converts routing decisions and simulated human reviews into two critical artifacts:

### 1️⃣ Audit Logs

Permanent records of **what decision happened and why**.

Example output:

```
task_id: T002
routing_decision: escalate
human_involved: true
final_decision: override_approved
decision_source: human
latency_seconds: 610
timestamp: 2026-03-04T18:00:00Z
```

---

### 2️⃣ Feedback Events

Signals that the **AI system may need improvement**.

Example:

```
task_id: T002
agent_correct: false
human_override: true
error_type: policy_edge_case
notes: Override: policy exception applied
```

These become **learning signals**.

---

# Why This Module Is Architecturally Important

Your orchestrator now models the **full enterprise decision loop**:

```
AI recommendation
↓
policy routing
↓
human oversight
↓
final decision
↓
audit logging
↓
feedback signal
```

This is essentially how **regulated industries deploy AI**.

Most AI demos stop here:

```
model.predict()
```

Your system continues through **governance and learning infrastructure**.

---

# Strengths in the Implementation

## 1️⃣ Clean Decision Mapping

Your helper function:

```python
_human_decision_to_final()
```

normalizes decisions into standardized outcomes.

Mapping:

```
approve  → approved
override → override_approved
modify   → modified_and_approved
reject   → rejected
```

This is excellent because **audit systems require normalized decision labels**.

Without this, reports become messy.

---

# 2️⃣ Single Audit Record Per Task

This is exactly what enterprise audit trails do.

Instead of logging events like:

```
task reviewed
task escalated
task approved
```

you produce a **final summarized decision record**.

This makes reporting much easier.

Example audit log:

```
task_id
risk_level
confidence_score
routing_decision
human_involved
final_decision
decision_source
latency_seconds
timestamp
```

This is **very clean schema design**.

---

# 3️⃣ Clear Decision Source

You explicitly record:

```
decision_source
```

Possible values:

```
agent
human
```

This is extremely important for:

```
AI governance
regulatory reporting
trust measurement
```

Executives often want to know:

```
What % of decisions required human intervention?
```

Your schema supports that easily.

---

# 4️⃣ Latency Simulation

You simulate processing time:

```
human decisions → slower
agent decisions → faster
```

Example:

```
agent latency ≈ 10–30 seconds
human latency ≈ 300–1200 seconds
```

This enables interesting analytics later:

```
human review bottlenecks
automation efficiency
cost of governance
```

That’s a **very nice design choice**.

---

# 5️⃣ Feedback Event Logic

Your feedback detection logic is simple and effective.

You create feedback when:

```
modify
reject
override
```

Meaning:

```
AI decision was not final
```

This is the correct signal for:

```
model improvement
policy tuning
workflow analysis
```

---

# 6️⃣ Human Override Detection

You correctly treat overrides differently:

```
override → policy_edge_case
modify/reject → human_adjustment
```

That distinction is important.

Overrides usually indicate:

```
policy limitations
edge cases
exceptions
```

While modifications indicate:

```
AI improvement opportunities
```

Very good separation.

---

# How This Fits Into the Final Pipeline

Your orchestrator now looks like this:

```
Goal
 ↓
Planning
 ↓
Data Loading
 ↓
Routing Policy Engine
 ↓
Human Simulator
 ↓
Audit Builder
 ↓
Feedback Builder
 ↓
Executive Report
```

Which produces:

```
audit_logs.json
feedback_events.json
```

This is **exactly how enterprise systems store decision trails**.

---

# Minor Improvements I Recommend

Nothing major — the architecture is already very good.

---

# 1️⃣ Deterministic Latency (Important)

Right now you use:

```python
hash(task_id)
```

The issue:

```
Python hash() is NOT deterministic between runs
```

Python intentionally randomizes hash seeds.

Meaning:

```
same task_id → different latency across runs
```

That breaks your **deterministic philosophy**.

Instead use your existing helper approach:

```
sha256(task_id)
```

Same pattern used in the simulator.

Example:

```
latency = deterministic_float(task_id) * range
```

---

# 2️⃣ Record the Triggering Rule (Highly Valuable)

You currently log:

```
routing_decision
```

But you should also log:

```
rule_id
```

Example:

```
rule_id: RULE_003
```

This allows executives to answer:

```
Which policy rules trigger the most escalations?
```

Very useful insight.

---

# 3️⃣ Include Reviewer ID in Audit Logs

You already store it in simulated reviews.

Adding it to audit entries would improve traceability.

Example:

```
reviewer_id
human_role
```

---

# 4️⃣ Record the Human Decision (Optional)

Right now audit logs show:

```
final_decision
```

But sometimes it's helpful to store:

```
human_decision
```

Example:

```
modify
override
reject
```

That gives richer analytics later.

---

# One Very Interesting Strategic Outcome

Because you store **feedback events**, you now have a dataset for:

```
AI error analysis
```

Example analysis:

```
tasks where human rejected AI
tasks where policy override occurred
domains with highest disagreement
```

You could later build an **AI Quality Monitoring Agent** using this dataset.

---

# Overall Assessment

This module completes your **enterprise decision infrastructure**.

Strengths:

```
clean audit schema
decision normalization
human override detection
feedback signal generation
latency modeling
```

Combined with the other modules you now have:

```
AI decision engine
policy governance
human oversight simulation
audit trail
feedback learning signals
```

That is essentially a **miniature enterprise AI governance stack**.

---

# One Big Picture Observation

Your orchestrator now models **four layers of trust**:

```
AI intelligence
policy governance
human oversight
audit transparency
```

That combination is exactly what executives want when deploying AI systems.

Very few agent projects include **all four**.




In [ ]:
"""Build audit log entries and feedback events from routing and simulated reviews."""

from datetime import datetime, timezone
from typing import Any, Dict, List


def _human_decision_to_final(human_decision: str, action: str) -> str:
    """Map human_decision to audit final_decision value."""
    if human_decision == "approve":
        return "approved"
    if human_decision == "override":
        return "override_approved"
    if human_decision == "modify":
        return "modified_and_approved"
    if human_decision == "reject":
        return "rejected"
    return "approved"


def build_audit_entries(
    routing_decisions: List[Dict[str, Any]],
    simulated_reviews: List[Dict[str, Any]],
    tasks_by_id: Dict[str, Dict[str, Any]],
    agent_output_by_task_id: Dict[str, Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """
    Build one audit log entry per task for this run.
    Includes routing_decision, human_involved, final_decision, decision_source, latency_seconds.
    """
    entries = []
    now = datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")
    review_lookup = {r["task_id"]: r for r in simulated_reviews}

    for dec in routing_decisions:
        task_id = dec.get("task_id")
        action = dec.get("action")
        human_involved = action in ("human_review", "escalate")
        review = review_lookup.get(task_id) if human_involved else None

        if human_involved and review:
            final_decision = _human_decision_to_final(review.get("human_decision", "approve"), action)
            decision_source = "human"
            # Simulated latency: 12–900 sec for auto, 120–1200 for human
            latency_seconds = 300 + (hash(task_id) % 900) if isinstance(hash(task_id), int) else 600
        else:
            final_decision = "approved"
            decision_source = "agent"
            latency_seconds = 10 + (hash(task_id) % 20) if isinstance(hash(task_id), int) else 12

        task = (tasks_by_id or {}).get(task_id) or {}
        out = (agent_output_by_task_id or {}).get(task_id) or {}
        entries.append({
            "task_id": task_id,
            "risk_level": task.get("risk_level"),
            "confidence_score": out.get("confidence_score"),
            "routing_decision": action,
            "human_involved": human_involved,
            "final_decision": final_decision,
            "decision_source": decision_source,
            "latency_seconds": latency_seconds,
            "timestamp": now,
        })
    return entries


def build_feedback_entries(
    simulated_reviews: List[Dict[str, Any]],
    routing_decisions: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """
    Build feedback events for tasks where human modified, rejected, or overrode.
    agent_correct=False when human changed the outcome; human_override=True for override.
    """
    entries = []
    now = datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")
    decision_by_task = {d["task_id"]: d for d in routing_decisions}

    for r in simulated_reviews:
        human_decision = r.get("human_decision")
        if human_decision not in ("modify", "reject", "override"):
            continue
        task_id = r.get("task_id")
        dec = decision_by_task.get(task_id) or {}
        entries.append({
            "task_id": task_id,
            "agent_correct": False,
            "human_override": human_decision == "override",
            "error_type": "policy_edge_case" if human_decision == "override" else "human_adjustment",
            "notes": r.get("human_feedback", ""),
            "timestamp": now,
        })
    return entries
